# 學生學習與作業版｜任務 11：資料切分與基準模型

**姓名：**何韋仁　**學號／組別：**C114118243　**完成日期：**2026/11/26

**學習目標：** 切分訓練、驗證、測試資料，建立多數類別基準並檢查資料分布。

本單元與教師版「任務 11」相同標題、資料、教學順序與範例；除了明確標示的作業與填空，其餘教師版教材內容保留。對照重點：程式填空對應教師版「11.2 多數類別基準」。

### 如何完成
1. 先閱讀觀念與執行資料準備；各單元可以獨立啟動，不需要依賴上一單元的變數。
2. **💻 P 作業**：在指定程式 cell 補上 `____`，可先用提供的 Vibe Coding 提示詞詢問 AI。補完再執行，並執行後面的驗證 cell。未填時出現 `NameError` 是預期現象，先完成填空再繼續。
3. **📝 Q 作業**：查詢 AI、比較教材內容，填寫緊接的文字作答區；請記錄實際提示詞、自己的解釋與查證方式。
4. **🎯 M 小任務**：在任務 4、8、12、16 將前面幾單元整合成可執行的成果。不要只交 AI 回覆，要交實際程式、輸出與自己的解釋。
5. 把 Notebook 存到自己的 Google 雲端硬碟，完成後下載／依教師指示繳交 `.ipynb`；小任務保留輸出、文字與 AI 使用紀錄。

**案例限制：**本資料是公式生成的 60 筆模擬紀錄，不能推論真實學生或因果。缺失與 15 小時值在計分後加入。預測時欄位可得性是假設；全部資料已在前半段探索，後續切出的測試集僅示範流程，不能視為完全獨立的新資料。

**完成檢核：**本單元有 1 項程式填空、2 項 AI 查詢作答；若是任務 4、8、12、16，另外有 1 項整合小任務。請保留自己的輸出、回答和驗證紀錄。


## 共用資料準備
以下在記憶體重建與既有教材相同的資料。執行後可以直接開始本單元；不會覆寫 CSV。


In [ ]:
# 匯入套件：numpy 建立隨機資料，pandas 管理表格，pyplot 繪圖
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 固定種子，並維持呼叫順序，以重現原教材的 60 筆模擬資料
rng = np.random.default_rng(42)
n = 60
departments = ["資訊管理", "企業管理", "金融管理"]
# 每個字典鍵是一個欄位；對應的值是 60 名學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)
# 成績依設定公式與隨機誤差建立；關係來自模擬規則，不能視為因果發現
noise = rng.normal(0, 5, n)
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)
# 故意在計分後加入資料品質問題；無法從表格還原計分時原始值
df.loc[[5, 17], "sleep_hours"] = np.nan
df.loc[8, "screen_time_hours"] = 15.0
print("模擬資料形狀（列, 欄）：", df.shape)


模擬資料形狀（列, 欄）： (60, 7)


## 機器學習共用準備
依標籤分層切成 60% 訓練、20% 驗證、20% 測試；建立流程時只在訓練集 `fit`。固定切分使後續任務可比較；本資料已探索過，測試分數不是嚴格的獨立評估。


In [ ]:
# 匯入資料切分、前處理、模型流程與評估工具
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

# 複製原始資料；將小考低於 75 分定義為教學用正類（1）
model_df = df.copy()
model_df["needs_support"] = (model_df["quiz_score"] < 75).astype(int)
numeric_columns = ["study_hours", "sleep_hours", "screen_time_hours", "attendance_rate"]
category_columns = ["department"]
feature_columns = numeric_columns + category_columns
# 明確選欄，避免把 quiz_score 或目標 needs_support 當成特徵
X = model_df[feature_columns].copy()
y = model_df["needs_support"].copy()
# 固定隨機種子；使用標籤分層，先保留 20% 測試集
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
# 剩下 80% 的四分之一為 20% 驗證集，最後得到 60/20/20
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)
# 數值缺失用訓練集中位數；類別缺失以訓練集眾數填補後做 One-Hot
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
category_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))])
# ColumnTransformer 指定各欄位的處理方式；fit 時僅從訓練集學習
preprocess = ColumnTransformer([("num", numeric_pipe, numeric_columns),
                                ("cat", category_pipe, category_columns)])
print("訓練／驗證／測試筆數：", len(X_train), len(X_val), len(X_test))


訓練／驗證／測試筆數： 36 12 12


## 11.1 切分後先檢查
**觀念說明：**訓練集用來學習，驗證集用來選方案，測試集保留到選定後。分層依標籤維持類別比例，但不能增加資料量。若預測未來，應按實際時間順序考慮切分；本資料沒有時間欄位。


In [ ]:
# 用索引檢查三組資料筆數、類別分布與互不重疊
for name, x_part, y_part in [("train", X_train, y_train), ("validation", X_val, y_val), ("test", X_test, y_test)]:
    print(name, "筆數", len(x_part), "類別", y_part.value_counts().sort_index().to_dict())
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_val) + len(X_test) == len(df)


train 筆數 36 類別 {0: 28, 1: 8}
validation 筆數 12 類別 {0: 9, 1: 3}
test 筆數 12 類別 {0: 9, 1: 3}


## 📝 學生作業 11-Q1｜AI 查詢與作答：三組資料用途
**任務：**訓練、驗證、測試資料分別做什麼？為何測試集不應反覆用來選模型？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請用考試類比解釋訓練、驗證與測試集，並說明多次選模對測試評估的影響。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 11-Q1 專用作答區：請雙擊此文字儲存格填寫
- **AI 工具及實際提示詞：** 使用 Claude，採用教材提供的提示詞。
- **AI 回答重點（請自行整理）：**
  1. 訓練：讓模型學習。
  2. 驗證：比較不同方案、選模型。
  3. 測試：選好之後最後檢查一次。
- **我的理解與具體例子：** 測試集如果一直拿來選，就等於偷看，最後分數會太樂觀。
- **查證來源／方式及日期（無法查證請註明）：** 2026/11/06 執行 11.1：訓練 36、驗證 12、測試 12，三組沒有重疊。
- **我檢查或修正 AI 回答的地方：** AI 沒提到這份資料前面已經全部看過，測試集也不是完全新的。

## 11.2 多數類別基準
訓練集中哪個類別最多，就一律預測該類；再在驗證集評估。準確率可能看似不錯，但正類 Recall 可能很低。


## 💻 學生作業 11-P1｜Vibe Coding 程式填空
**對應教師版：**任務 11「11.2 多數類別基準」的示範程式。

**任務：**讓 baseline 僅以 X_train 與 y_train 學習，再由後續程式預測驗證資料。

**執行方式：**找到下一個程式儲存格中的 `____`，使用自己撰寫或 AI 建議的程式補齊，保留其他已提供的程式與註解。補完後先執行該 cell，再執行緊接的自動檢查；後續單元內容可能依賴此變數。

**可使用的 Vibe Coding 提示詞：**
```text
我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。
我的任務是：讓 baseline 僅以 X_train 與 y_train 學習，再由後續程式預測驗證資料。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
```
完成後記錄實際使用的提示詞與自己改動的地方。


In [ ]:
# DummyClassifier 的 most_frequent 只從訓練標籤找出多數類別
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)  # 只用訓練集
base_val_pred = baseline.predict(X_val)
print("驗證 Accuracy：", accuracy_score(y_val, base_val_pred))
print("驗證正類 Recall：", recall_score(y_val, base_val_pred, zero_division=0))
print("混淆矩陣（列實際、欄預測；順序 0,1）：\n", confusion_matrix(y_val, base_val_pred, labels=[0,1]))

驗證 Accuracy： 0.75
驗證正類 Recall： 0.0
混淆矩陣（列實際、欄預測；順序 0,1）：
 [[9 0]
 [3 0]]


### ✅ 作業 11-P1｜執行驗證
執行下方程式，並同時檢查上方實際輸出。自動檢查只涵蓋主要條件；如果圖表、說明或資料來源不符任務，仍需人工修正。


In [ ]:
assert hasattr(baseline, "classes_")
assert len(base_val_pred) == len(y_val)
print("檢查通過：基準已訓練且預測筆數正確。")


檢查通過：基準已訓練且預測筆數正確。


### ✍️ 作業 11-P1｜我的完成紀錄
- **我實際使用的提示詞（若未使用 AI 請註明）：** 使用 Claude，採用教材提供的提示詞。
- **我補上的程式在做什麼：** `baseline.fit(X_train, y_train)`，只用訓練集找出最多的類別。
- **執行或驗證遇到的問題與修正：** 沒有遇到錯誤。
- **檢查結果及我如何確認：** 驗證 Accuracy 0.75，但 Recall 0，3 個正類全部漏掉。

## 📝 學生作業 11-Q2｜AI 查詢與作答：時間與泛化
**任務：**如果目標是預測未來，為何常依時間切分？本案例為何做不到？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請比較時間切分與隨機切分，並說明沒有時間欄位時不可假稱已進行時間驗證。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 11-Q2 專用作答區：請雙擊此文字儲存格填寫
- **AI 工具及實際提示詞：** 使用 Claude，採用教材提供的提示詞。
- **AI 回答重點（請自行整理）：**
  1. 預測未來要用過去訓練、未來驗證，比較接近實際。
  2. 隨機切分可能讓模型看到未來的資料。
- **我的理解與具體例子：** 這份資料沒有時間欄位，所以只能隨機切，沒辦法照時間切。
- **查證來源／方式及日期（無法查證請註明）：** 2026/11/06 確認 df 欄位中沒有日期或時間。
- **我檢查或修正 AI 回答的地方：** AI 沒提到同一個人出現在不同組也會有問題。

## 11.3 獨立資料與時間切分
整份資料已在任務 1～8 被查看，後續測試集只是流程教學示範，不是完全獨立的評估。部署前需新收集、未參與分析與設計的新樣本。

**思考：**每日銷售資料若要預測未來，為何較早訓練、較晚驗證通常比隨機打亂更貼近情境？重複客戶還要檢查跨資料集是否共用同一人的資訊。


## 延伸：分層比例與小樣本
比例接近不代表每組都有足夠樣本；實際列出類別數量，尤其是驗證及測試中的正類數。時間資料也不能為了分層而任意打亂未來。


In [ ]:
# 三組的正類筆數及比例，協助解讀評估指標
for label, target in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(label, "總數", len(target), "正類", int(target.sum()), "正類比例", round(target.mean(),3))


train 總數 36 正類 8 正類比例 0.222
validation 總數 12 正類 3 正類比例 0.25
test 總數 12 正類 3 正類比例 0.25


## 本單元學習收束與 AI 使用紀錄

請學生先用自己的話解釋一個關鍵觀念，再指出一個執行結果及驗證方式。記錄：使用的提示詞、AI 回答中檢查或修改的地方、遇到的問題、修正後的證據與本資料限制。

**通用提示詞**：我是 Python 初學者，請逐步協助我完成上面的任務；每段程式加繁體中文註解，說明輸入、處理與輸出，提供兩項可實際執行的驗證，保留原始資料，不要虛構執行結果。
